# Pemeriksaan terpisah FGD-6

Notebook ini memeriksa struktur dan bagian training FGD-6 secara lokal. Pemeriksaan tetap di sini; notebook utama hanya berisi alur training.

### Baca notebook training

In [1]:
import ast
import json
from pathlib import Path
from types import SimpleNamespace

training_path = next(
    path for path in [Path("FGD_6.ipynb"), Path("source code/FGD_6.ipynb")]
    if path.is_file()
)
notebook = json.loads(training_path.read_text(encoding="utf-8"))
sources = {
    cell["id"]: "".join(cell["source"])
    for cell in notebook["cells"] if cell["cell_type"] == "code"
}

### Periksa struktur training

In [2]:
import nbformat

nbformat.validate(nbformat.from_dict(notebook))
assert list(sources) == [
    "fgd6_03", "fgd6_05", "fgd6_07", "fgd6_09", "fgd6_11", "fgd6_13",
]
for cell_id, source in sources.items():
    tree = ast.parse(source)
    assert not any(isinstance(node, (ast.Assert, ast.Raise, ast.Try)) for node in ast.walk(tree)), cell_id
assert sum(source.count("model.fit(") for source in sources.values()) == 2
assert 'datasets["DenseNet121"]["test"]' not in "".join(sources.values())
assert "config = {" not in "".join(sources.values())
print("Lulus: enam cell training, sintaks valid, tanpa test atau kode audit di notebook utama.")

Lulus: enam cell training, sintaks valid, tanpa test atau kode audit di notebook utama.


### Periksa pembukaan blok akhir

In [3]:
class BatchNormalization:
    def __init__(self, name):
        self.name = name
        self.trainable = True


layers = [
    SimpleNamespace(name="conv4_block24_1_conv", trainable=True),
    SimpleNamespace(name="conv5_block1_1_conv", trainable=False),
    BatchNormalization("conv5_block1_0_bn"),
    BatchNormalization("bn"),
]
calls = []
namespace = {
    "warmup_checkpoint": "warmup_best.weights.h5",
    "base_model": SimpleNamespace(layers=layers, trainable=False),
    "model": SimpleNamespace(
        load_weights=lambda path: calls.append(("load", path)),
        compile=lambda **kwargs: calls.append(("compile", kwargs)),
    ),
    "tf": SimpleNamespace(keras=SimpleNamespace(
        layers=SimpleNamespace(BatchNormalization=BatchNormalization),
        optimizers=SimpleNamespace(Adam=lambda **kwargs: SimpleNamespace(**kwargs)),
    )),
}
exec(sources["fgd6_09"], namespace)
assert namespace["base_model"].trainable
assert [layer.trainable for layer in layers] == [False, True, False, False]
assert [call[0] for call in calls] == ["load", "compile"]
assert calls[1][1]["optimizer"].learning_rate == 0.00001
print("Lulus: hanya blok conv5 non-BatchNormalization yang terbuka dengan optimizer fine-tuning baru.")

Lulus: hanya blok conv5 non-BatchNormalization yang terbuka dengan optimizer fine-tuning baru.


### Periksa pemilihan checkpoint

In [4]:
for warmup_loss, finetune_loss, expected_stage in [
    (0.2, 0.1, "fine-tuning"),
    (0.1, 0.2, "warm-up"),
    (0.1, 0.1, "warm-up"),
]:
    loaded = []
    namespace = {
        "history_warmup": SimpleNamespace(history={"val_loss": [warmup_loss]}),
        "history_finetune": SimpleNamespace(history={"val_loss": [finetune_loss]}),
        "warmup_checkpoint": "warmup_best.weights.h5",
        "finetune_checkpoint": "finetune_best.weights.h5",
        "model": SimpleNamespace(load_weights=lambda path: loaded.append(path)),
        "print": lambda *args, **kwargs: None,
    }
    exec(sources["fgd6_13"], namespace)
    assert namespace["selected_stage"] == expected_stage
    assert loaded == [namespace["selected_checkpoint"]]
print("Lulus: checkpoint terbaik dipilih; jika val_loss seri, warm-up dipakai.")

Lulus: checkpoint terbaik dipilih; jika val_loss seri, warm-up dipakai.
